# Ames Housing: Làm sạch dữ liệu & Trực quan hóa (EDA)

Notebook này được xây dựng **trên nền code bạn đã viết**. Để dễ đối chiếu, mỗi cell code có chú thích đầu cell:

- `# [GIỮ]` – code của bạn, giữ nguyên
- `# [SỬA]` – code của bạn nhưng có lỗi hoặc chưa phù hợp, đã sửa (kèm lý do)
- `# [THÊM]` – phần còn thiếu so với đề bài

**Cấu trúc:** Phần 1 – Tiền xử lý (Profiling → Missing Values → Outliers); Phần 2 – EDA với 5 biểu đồ chuẩn publication, mỗi biểu đồ kèm *Insights*.

**Cách chạy:** đặt file CSV cùng thư mục với notebook, cài thư viện (`pip install pandas numpy matplotlib seaborn scipy scikit-learn missingno`) rồi chạy lần lượt từ trên xuống. 5 ảnh được lưu trong thư mục `figures/`.

## 0. Thiết lập môi trường

In [ ]:
# [GIỮ] import của bạn
import pandas as pd
import matplotlib.pyplot as plt
import missingno as msno   # cần: pip install missingno

In [ ]:
import glob, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# --- Bảng màu Okabe-Ito: được thiết kế để thân thiện với người mù màu ---
OI = dict(black="#000000", orange="#E69F00", sky="#56B4E9", green="#009E73",
          yellow="#F0E442", blue="#0072B2", verm="#D55E00", pink="#CC79A7",
          grey="#999999", lightgrey="#CFCFCF")

# --- Style chung cho mọi biểu đồ: font đủ lớn, nền trắng, bỏ lưới thừa ---
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300,
    "font.size": 12, "axes.titlesize": 14, "axes.titleweight": "bold",
    "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11,
    "legend.fontsize": 11, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False,   # bỏ spines không cần thiết
    "axes.grid": True, "grid.color": "#E6E6E6", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "figure.facecolor": "white", "axes.facecolor": "white",
})

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(fig, name):
    """Lưu ảnh độ phân giải cao (300 dpi), cắt sát lề."""
    fig.savefig(os.path.join(FIG_DIR, name), dpi=300, bbox_inches="tight", facecolor="white")

# Định dạng trục giá: 150000 -> $150K
price_fmt = mtick.FuncFormatter(lambda x, _: f"${x/1000:,.0f}K")

In [ ]:
# [GIỮ] import của bạn (đặt ở đây để các cell Imputer phía sau dùng được)
from sklearn.impute import SimpleImputer

# PHẦN 1. TIỀN XỬ LÝ DỮ LIỆU (DATA CLEANING)

## 1.1 Khám phá dữ liệu (Data Profiling)

In [ ]:
# [SỬA] Không hard-code tên file: tự tìm file có chữ "Ames" trong thư mục hiện tại
# (tên file của bạn "AmesHousing - AmesHousing.csv.csv" vẫn được nhận)
path = sorted(glob.glob("*Ames*.csv*"))[0]
df = pd.read_csv(path)
print("Đã đọc:", path)

**Ma trận khuyết thiếu (missingno).** Cả 3 biểu đồ dưới là của bạn; chỉ sửa cách đặt kích thước: `msno` tự tạo figure riêng, nên `plt.figure(figsize=...)` đứng trước sẽ sinh ra một khung trắng thừa. Cần truyền `figsize` trực tiếp vào hàm `msno`.

In [ ]:
# [SỬA] truyền figsize vào msno thay vì plt.figure(); dùng ax.set_title thay cho plt.title
ax = msno.matrix(df, figsize=(14, 6), sparkline=True, fontsize=8)
ax.set_title("Ma Trận Khảo Sát Giá Trị Khuyết (Nullity Matrix)", fontsize=14)
plt.show()

In [ ]:
# [SỬA] cùng lý do: truyền figsize; đổi màu sang xanh dương Okabe-Ito (thân thiện người mù màu)
ax = msno.bar(df, color=OI["blue"], fontsize=9, figsize=(14, 6))
ax.set_title("Tỷ Lệ Hoàn Thiện Dữ Liệu Theo Cột", fontsize=14)
plt.show()

In [ ]:
# [SỬA] truyền figsize; heatmap chỉ hiển thị các cột có khuyết thiếu
ax = msno.heatmap(df, cmap="coolwarm", figsize=(11, 8), fontsize=9)
ax.set_title("Bản Đồ Nhiệt Tương Quan Khuyết Thiếu (Nullity Correlation)", fontsize=14)
plt.show()

In [ ]:
# [GIỮ]
df.info()

In [ ]:
# [GIỮ]
df.head()

In [ ]:
# [GIỮ]
df.tail()

In [ ]:
# [GIỮ]
df.sample(5)

In [ ]:
# [GIỮ] kích thước (hàng, cột)
df.shape

In [ ]:
# [GIỮ]
df.columns

In [ ]:
# [GIỮ]
df.dtypes

In [ ]:
# [GIỮ] tổng số ô khuyết thiếu theo từng cột
df.isnull().sum()

In [ ]:
# [GIỮ] tổng số ô khuyết thiếu toàn bộ dữ liệu
df.isnull().sum().sum()

In [ ]:
# [SỬA] bản gốc in cả 82 cột (kể cả cột đầy đủ) rất dài; chỉ in cột có thiếu, sắp xếp giảm dần
pct_missing = df.isnull().mean() * 100
print("The percentage of missing data:")
print(pct_missing[pct_missing > 0].sort_values(ascending=False).round(2).to_string())

In [ ]:
# [GIỮ] số dòng trùng lặp
df.duplicated().sum()

In [ ]:
# [GIỮ]
df.describe()

In [ ]:
# [THÊM] Bảng tổng hợp theo yêu cầu 1.1: kích thước, kiểu dữ liệu, tỷ lệ khuyết thiếu (gộp trong 1 bảng)
print("Kích thước (hàng, cột):", df.shape)
print("Số cột theo kiểu dữ liệu:")
print(df.dtypes.value_counts().to_string())
print("Số PID trùng lặp:", df["PID"].duplicated().sum())

profile = pd.DataFrame({
    "dtype": df.dtypes,
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
missing_tbl = profile[profile["n_missing"] > 0].sort_values("pct_missing", ascending=False)
print(f"Có {len(missing_tbl)}/{df.shape[1]} cột bị khuyết thiếu")
display(missing_tbl)

**Nhận xét ban đầu**
- Bộ dữ liệu có 2.930 dòng × 82 cột, không có dòng trùng lặp; 27 cột có khuyết thiếu.
- Theo **từ điển dữ liệu của Ames**, phần lớn "khuyết thiếu" thực chất là **nhà không có tiện ích đó**, không phải mất dữ liệu: `NA` ở `Pool QC` = không có hồ bơi, `Alley` = không có ngõ, `Garage *` = không có gara, `Bsmt *` = không có tầng hầm... Ngoài ra `pandas` mặc định đọc chuỗi `"None"` (ví dụ `Mas Vnr Type = None`, không ốp gạch/đá) thành `NaN`.
- Vì vậy **không thể** điền mean/mode máy móc cho mọi cột; cần phân loại từng nhóm theo ý nghĩa nghiệp vụ (mục 1.2).

## 1.2 Xử lý dữ liệu khuyết thiếu (Missing Values)

### 1.2.1 Phương án ban đầu của bạn: mean cho biến số, mode cho biến phân loại
Hai cell dưới là code của bạn, chạy đúng và rất tiện làm **baseline**. Tuy nhiên có 2 điểm cần lưu ý trước khi dùng thật (kiểm chứng ngay ở cell sau):
1. `df_mean` và `df_mode` là hai bảng tách rời, chưa được ghép lại thành một DataFrame hoàn chỉnh.
2. Điền mode cho **mọi** cột phân loại là sai về nghiệp vụ: `Pool QC` thiếu 99,6% nhưng sẽ bị điền thành một mức chất lượng hồ bơi duy nhất cho gần 2.900 nhà **không có hồ bơi**.

In [ ]:
# [GIỮ] mean imputation cho biến số
numeric = df.select_dtypes(include="number")
mean_imputer = SimpleImputer(strategy="mean")
df_mean = pd.DataFrame(mean_imputer.fit_transform(numeric), columns=numeric.columns)

In [ ]:
# [GIỮ] kiểm tra: số ô khuyết trước và sau khi điền mean
print(numeric.isnull().sum().sum(), "->", df_mean.isnull().sum().sum())

In [ ]:
# [SỬA] select_dtypes(include=["object","category","string"]) phụ thuộc phiên bản pandas
# (pandas mới dùng dtype "str"); cách chắc chắn nhất là lấy mọi cột KHÔNG phải số
category = df.select_dtypes(exclude="number")
mode_imputer = SimpleImputer(strategy="most_frequent")
df_mode = pd.DataFrame(mode_imputer.fit_transform(category), columns=category.columns)
print(category.isnull().sum().sum(), "->", df_mode.isnull().sum().sum())

In [ ]:
# [THÊM] Kiểm chứng vì sao baseline chưa phù hợp
print("Pool QC thiếu:", round(df["Pool QC"].isna().mean() * 100, 2), "%  (nhà không có hồ bơi)")
print("Sau mode-imputation, Pool QC của", df_mode["Pool QC"].notna().sum(), "nhà trở thành:")
print(df_mode["Pool QC"].value_counts().to_string())
print()
print("Garage Yr Blt: có", df["Garage Yr Blt"].isna().sum(), "nhà không có gara; mean-imputation gán cho tất cả năm",
      round(df["Garage Yr Blt"].mean()), "(một năm xây gara không hề tồn tại)")

### 1.2.2 Chiến lược theo từng nhóm cột

| Nhóm | Cột | Tỷ lệ thiếu | Quyết định | Lý do |
|---|---|---|---|---|
| A. Thiếu cực nhiều | `Pool QC`, `Misc Feature`, `Alley` | 93–99,6% | **Drop** | Gần như hằng số, không giúp phân biệt giá, dễ gây overfit |
| B. Thiếu = không có tiện ích (categorical) | `Fence`, `Fireplace Qu`, `Mas Vnr Type`, `Garage Type/Finish/Qual/Cond`, `Bsmt Qual/Cond/Exposure`, `BsmtFin Type 1/2` | 2,7–80% | **Impute `"None"`** | Giữ "không có" như một hạng mục riêng vì chính việc không có gara/hầm/lò sưởi đã ảnh hưởng đến giá |
| C. Thiếu = không có tiện ích (numeric) | `Mas Vnr Area`, `BsmtFin SF 1/2`, `Bsmt Unf SF`, `Total Bsmt SF`, `Bsmt Full/Half Bath`, `Garage Area`, `Garage Cars` | <1% | **Impute 0** | Không có hầm/gara thì diện tích và số lượng đúng bằng 0 |
| D. Thiếu ngẫu nhiên, numeric | `Lot Frontage` | 16,7% | **Median theo `Neighborhood`** | Mặt tiền lô đất tương đồng trong cùng khu phố; median bền với ngoại lai hơn mean |
| E. Thiếu có cấu trúc | `Garage Yr Blt` | 5,4% | **Impute bằng `Year Built`** | Tránh một năm xây gara giả; gara thường xây cùng thời điểm với nhà |
| F. Thiếu rất ít, categorical | `Electrical` | 1 dòng | **Mode** | Chỉ 1 dòng, mode (`SBrkr`) chiếm đa số tuyệt đối |

`Order` và `PID` chỉ là mã định danh nên bị bỏ.

In [ ]:
# [THÊM] Áp dụng chiến lược trên; df_clean là bản sạch, df (gốc) giữ nguyên để đối chiếu
df_clean = df.copy()

# 0) Bỏ cột định danh
df_clean = df_clean.drop(columns=["Order", "PID"])

# A) Drop các cột thiếu > 90%
df_clean = df_clean.drop(columns=["Pool QC", "Misc Feature", "Alley"])

# B) Categorical: NaN nghĩa là "không có tiện ích" -> hạng mục "None"
none_cols = ["Fence", "Fireplace Qu", "Mas Vnr Type",
             "Garage Type", "Garage Finish", "Garage Qual", "Garage Cond",
             "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]
df_clean[none_cols] = df_clean[none_cols].fillna("None")

# C) Numeric: không có tiện ích -> 0
zero_cols = ["Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF", "Total Bsmt SF",
             "Bsmt Full Bath", "Bsmt Half Bath", "Garage Area", "Garage Cars"]
df_clean[zero_cols] = df_clean[zero_cols].fillna(0)

# D) Lot Frontage: median theo Neighborhood (fallback median toàn bộ nếu cả khu phố đều thiếu)
df_clean["Lot Frontage"] = df_clean.groupby("Neighborhood")["Lot Frontage"].transform(lambda s: s.fillna(s.median()))
df_clean["Lot Frontage"] = df_clean["Lot Frontage"].fillna(df_clean["Lot Frontage"].median())

# E) Garage Yr Blt: nhà không có gara -> dùng Year Built
df_clean["Garage Yr Blt"] = df_clean["Garage Yr Blt"].fillna(df_clean["Year Built"])
# Sửa lỗi nhập liệu: năm xây gara không thể lớn hơn năm bán (vd. có giá trị 2207)
bad_year = df_clean["Garage Yr Blt"] > df_clean["Yr Sold"]
print("Số dòng Garage Yr Blt > Yr Sold (lỗi nhập liệu):", bad_year.sum())
df_clean.loc[bad_year, "Garage Yr Blt"] = df_clean.loc[bad_year, "Year Built"]

# F) Electrical: điền mode
df_clean["Electrical"] = df_clean["Electrical"].fillna(df_clean["Electrical"].mode()[0])

# Kiểm tra
print("Tổng số giá trị khuyết thiếu còn lại:", int(df_clean.isna().sum().sum()))
print("Kích thước sau khi xử lý:", df_clean.shape)

**Kiểm tra:** không còn `NaN` (82 → 77 cột). Việc phân biệt "thiếu thông tin" với "không có tiện ích" giữ lại được tín hiệu quý cho mô hình thay vì xóa nhầm cột hay điền trung bình vô nghĩa.

### 1.2.3 (Tùy chọn) Pipeline tái sử dụng cho giai đoạn Machine Learning
Pipeline `ColumnTransformer` trong bản gốc của bạn là hướng đi đúng cho bước ML sau này (fit trên train, áp sang test để **không rò rỉ dữ liệu**). Có 3 điểm đã sửa: (1) `add_indicator=True` đẻ thêm hàng trăm cột cờ chưa cần; (2) `median`/`most_frequent` cho mọi cột lại lặp lại sai lầm ở 1.2.1; (3) `fit_transform` ra `numpy` làm mọi cột thành `object`, mất kiểu dữ liệu. Phiên bản dưới áp **cùng chiến lược theo nhóm** như trên và giữ nguyên dtype (`set_output`). Cell này chỉ minh họa, phần phân tích tiếp theo vẫn dùng `df_clean`.

In [ ]:
# [SỬA] Pipeline theo nhóm cột (dựa trên ColumnTransformer của bạn)
from sklearn.compose import ColumnTransformer
import numpy as np

X = df.drop(columns=["Order", "PID", "Pool QC", "Misc Feature", "Alley"])

preprocessor = ColumnTransformer(
    transformers=[
        ("none",   SimpleImputer(strategy="constant", fill_value="None"), none_cols),
        ("zero",   SimpleImputer(strategy="constant", fill_value=0),      zero_cols),
        ("median", SimpleImputer(strategy="median"),                      ["Lot Frontage", "Garage Yr Blt"]),
        ("mode",   SimpleImputer(strategy="most_frequent"),               ["Electrical"]),
    ],
    remainder="passthrough",            # các cột còn lại không thiếu -> giữ nguyên
    verbose_feature_names_out=False,
).set_output(transform="pandas")        # trả về DataFrame, giữ dtype

df_imputed = preprocessor.fit_transform(X)
print("NaN còn lại:", int(df_imputed.isna().sum().sum()), "| shape:", df_imputed.shape)
print(df_imputed.dtypes.value_counts().to_string())

## 1.3 Xử lý ngoại lai (Outliers)

Bạn đã chọn đúng hướng: so sánh **3 phương pháp** (IQR, Z-score, Modified Z-score/MAD). Code gốc có 4 lỗi cần sửa, nếu giữ nguyên sẽ cho kết quả sai:

1. `df[(numeric < lower) | (numeric > upper)]` dùng mặt nạ 2 chiều nên trả về **toàn bộ 2.930 dòng** (ô không phải ngoại lai thành `NaN`) → `len()` luôn bằng số dòng. Phải dùng `.any(axis=1)` để đếm dòng chứa ít nhất một giá trị bất thường.
2. `mad = np.median(np.abs(numeric - median))` lấy median của **toàn bộ bảng** thành một con số duy nhất, trong khi MAD phải tính **riêng cho từng cột**.
3. Nhiều cột có MAD = 0 (ví dụ `Pool Area`, `3Ssn Porch`: đa số bằng 0) → chia cho 0 ra `inf`; cần loại các cột này khỏi Modified Z-score.
4. `numeric` chứa cả `Order`, `PID`, `MS SubClass`, `Mo Sold`... là mã/nhãn dạng số, không có ý nghĩa để đi tìm ngoại lai. Mình chỉ giữ các biến diện tích/giá liên tục.

Dòng in `(Bị Masking!)` / `(Kháng Masking thành công)` cũng được bỏ: kết luận phải đến từ số liệu in ra, không viết trước.

In [ ]:
# [SỬA] Chỉ xét biến liên tục (diện tích, giá); bỏ các cột ID / biến phân loại mã hóa bằng số
cont_cols = ["Lot Frontage", "Lot Area", "Mas Vnr Area", "BsmtFin SF 1", "BsmtFin SF 2", "Bsmt Unf SF",
             "Total Bsmt SF", "1st Flr SF", "2nd Flr SF", "Low Qual Fin SF", "Gr Liv Area", "Garage Area",
             "Wood Deck SF", "Open Porch SF", "Enclosed Porch", "3Ssn Porch", "Screen Porch",
             "Pool Area", "Misc Val", "SalePrice"]
numeric = df_clean[cont_cols]          # dùng df_clean (đã hết NaN) thay cho df gốc
print(numeric.columns)

In [ ]:
# [GIỮ]
numeric.describe()

In [ ]:
# [SỬA] Phương pháp 1: IQR (Tukey). Mặt nạ boolean 2 chiều + .any(axis=1) để lấy các DÒNG bất thường
Q1 = numeric.quantile(0.25)
Q3 = numeric.quantile(0.75)
IQR = Q3 - Q1
iqr_lower = Q1 - 1.5 * IQR
iqr_upper = Q3 + 1.5 * IQR
iqr_mask = (numeric < iqr_lower) | (numeric > iqr_upper)
iqr_outliers = df_clean[iqr_mask.any(axis=1)]

In [ ]:
# [SỬA] Phương pháp 2: Z-Score truyền thống
mean = numeric.mean()
std = numeric.std()
z_scores = (numeric - mean) / std
z_mask = z_scores.abs() > 3
z_outliers = df_clean[z_mask.any(axis=1)]

In [ ]:
# [SỬA] Phương pháp 3: Modified Z-Score (MAD) – MAD tính RIÊNG TỪNG CỘT (axis=0), bỏ cột có MAD = 0
median = numeric.median()
mad = (numeric - median).abs().median()
print("Cột có MAD = 0 (bị loại khỏi phương pháp này):", list(mad[mad == 0].index))
mad = mad.replace(0, np.nan)
modified_z_scores = 0.6745 * (numeric - median) / mad
modz_mask = modified_z_scores.abs() > 3.5
mod_z_outliers = df_clean[modz_mask.any(axis=1)]

In [ ]:
# [SỬA] So sánh kết quả: số DÒNG bất thường (bản gốc luôn in 2930 do lỗi mặt nạ) và số ngoại lai theo từng cột
print("=== KẾT QUẢ SO SÁNH PHÁT HIỆN NGOẠI LAI (số dòng có ít nhất 1 giá trị bất thường) ===")
print(f"IQR (1.5*IQR)         : {len(iqr_outliers)} dòng")
print(f"Z-Score (|Z| > 3)     : {len(z_outliers)} dòng")
print(f"Modified Z (|M| > 3.5): {len(mod_z_outliers)} dòng")

summary = pd.DataFrame({"IQR": iqr_mask.sum(), "Z-score": z_mask.sum(), "Modified Z": modz_mask.sum()})
display(summary)

**Đọc kết quả:** mỗi phương pháp gắn cờ một số lượng rất khác nhau, và IQR gắn cờ hơn một nửa số dòng chỉ vì phân phối lệch phải và quá nhiều giá trị 0 của các biến diện tích. Không thể xóa hết: đó là nhà thật. Vấn đề là các điểm **bất thường trong quan hệ giữa hai biến** (diện tích lớn nhưng giá thấp), mà kiểm tra từng biến riêng lẻ không bắt được. Đề bài cũng nhấn mạnh điểm này, nên ta xem biểu đồ.

In [ ]:
# [SỬA] Bản gốc dùng x="category", y="number" (chuỗi mẫu, chạy sẽ lỗi). Thay bằng cột thật:
# phân bố Gr Liv Area theo Overall Qual qua 3 kiểu biểu đồ stripplot / boxplot / violinplot
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
sns.stripplot(data=df_clean, x="Overall Qual", y="Gr Liv Area", jitter=True, size=2.5, alpha=0.5, color=OI["blue"], ax=axes[0])
axes[0].set_title("Stripplot")
sns.boxplot(data=df_clean, x="Overall Qual", y="Gr Liv Area", color=OI["sky"], fliersize=2.5, ax=axes[1])
axes[1].set_title("Boxplot")
sns.violinplot(data=df_clean, x="Overall Qual", y="Gr Liv Area", color=OI["sky"], inner="quartile", cut=0, ax=axes[2])
axes[2].set_title("Violinplot")
for a in axes:
    a.set_xlabel("Chất lượng tổng thể (Overall Qual)")
axes[0].set_ylabel("Gr Liv Area (sq ft)")
axes[1].set_ylabel(""); axes[2].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# [THÊM] Ngoại lai đa biến theo gợi ý của đề bài: diện tích lớn nhưng giá thấp bất thường
print(df_clean[df_clean["Gr Liv Area"] > 4000][["Gr Liv Area", "SalePrice", "Sale Condition", "Neighborhood", "Overall Qual"]])

outlier_mask = (df_clean["Gr Liv Area"] > 4000) & (df_clean["SalePrice"] < 300_000)
print("Số điểm bị loại:", outlier_mask.sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, data, title in [(axes[0], df_clean, "Trước khi loại ngoại lai"),
                        (axes[1], df_clean[~outlier_mask], "Sau khi loại ngoại lai")]:
    ax.scatter(data["Gr Liv Area"], data["SalePrice"], s=14, alpha=0.5, color=OI["blue"], edgecolor="none")
    ax.set_title(title); ax.set_xlabel("Diện tích sinh hoạt trên mặt đất (sq ft)")
    ax.yaxis.set_major_formatter(price_fmt)
axes[0].set_ylabel("Giá bán")
axes[0].scatter(df_clean.loc[outlier_mask, "Gr Liv Area"], df_clean.loc[outlier_mask, "SalePrice"],
                s=90, facecolor="none", edgecolor=OI["verm"], linewidth=2, label="Ngoại lai bị loại")
axes[0].legend(loc="upper left")
plt.tight_layout(); plt.show()

# Hệ số tương quan Gr Liv Area - SalePrice trước và sau khi loại
r_before = df_clean["Gr Liv Area"].corr(df_clean["SalePrice"])
r_after = df_clean.loc[~outlier_mask, "Gr Liv Area"].corr(df_clean.loc[~outlier_mask, "SalePrice"])
print(f"Tương quan Gr Liv Area - SalePrice: {r_before:.3f} -> {r_after:.3f}")

# Áp dụng loại bỏ
df_clean = df_clean[~outlier_mask].reset_index(drop=True)
print("Kích thước dữ liệu sạch:", df_clean.shape)

**Quyết định xử lý ngoại lai**
- **Loại 3 điểm** có `Gr Liv Area` > 4.000 sq ft nhưng giá < 300.000$. Cả 3 đều là giao dịch `Partial` (nhà mới xây, bán khi chưa hoàn thiện) nên giá không phản ánh giá trị thật; chúng kéo đường xu hướng xuống và làm giảm tương quan diện tích–giá.
- **Giữ 2 nhà lớn giá 745–755 nghìn $:** nằm đúng xu hướng "lớn hơn → đắt hơn", là *giá trị cực đoan hợp lệ*, không phải lỗi dữ liệu.
- **Không cắt các ngoại lai IQR / Z-score còn lại** (`Lot Area`, `Garage Area`, `SalePrice` cao...): đó là nhà thật, đặc điểm tự nhiên của thị trường (phân phối lệch phải). Loại bỏ sẽ mất thông tin phân khúc cao cấp; thay vào đó xử lý bằng **biến đổi log** (Biểu đồ 1).

In [ ]:
# [THÊM] Lưu dữ liệu sạch để dùng lại ở bước Machine Learning (tên file không chứa chữ "Ames" để khỏi bị đọc nhầm)
df_clean.to_csv("housing_clean.csv", index=False)

# Từ đây Phần 2 dùng biến df = dữ liệu đã làm sạch
df = df_clean.copy()
print("Dữ liệu cho Phần 2:", df.shape)

# PHẦN 2. TRỰC QUAN HÓA DỮ LIỆU (EDA) – CHUẨN PUBLICATION

## Biểu đồ 1 – Phân phối biến mục tiêu `SalePrice`

In [ ]:
price = df["SalePrice"]
price_log = np.log1p(price)            # log(1 + x): ổn định với giá trị nhỏ, kéo đuôi phải về gần phân phối chuẩn

# Thống kê trước/sau biến đổi
stat = lambda s: dict(skew=stats.skew(s), kurtosis=stats.kurtosis(s))
st_raw, st_log = stat(price), stat(price_log)
print(f"SalePrice gốc : skew = {st_raw['skew']:.3f}, kurtosis = {st_raw['kurtosis']:.3f}")
print(f"log1p(SalePrice): skew = {st_log['skew']:.3f}, kurtosis = {st_log['kurtosis']:.3f}")
print(f"Mean = {price.mean():,.0f} | Median = {price.median():,.0f}")

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

def hist_panel(ax, s, color, title, xlabel, st, fmt=None):
    """Histogram + KDE + đường mean/median để thấy độ lệch."""
    sns.histplot(s, bins=40, kde=True, color=color, edgecolor="white", linewidth=0.5, ax=ax,
                 line_kws={"linewidth": 2})
    ax.axvline(s.mean(), color=OI["black"], linestyle="--", linewidth=1.5, label=f"Mean")
    ax.axvline(s.median(), color=OI["verm"], linestyle=":", linewidth=2, label=f"Median")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Số lượng nhà")
    ax.legend(loc="upper right")
    ax.text(0.97, 0.62, f"Skewness = {st['skew']:.2f}\nKurtosis = {st['kurtosis']:.2f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec=OI["lightgrey"]))
    if fmt: ax.xaxis.set_major_formatter(fmt)

hist_panel(axes[0, 0], price, OI["sky"], "(a) SalePrice gốc: lệch phải", "Giá bán", st_raw, price_fmt)
hist_panel(axes[0, 1], price_log, OI["green"], "(b) Sau biến đổi log(1 + SalePrice)", "log(1 + Giá bán)", st_log)

def qq_panel(ax, s, color, title):
    """Q-Q plot: điểm càng bám sát đường chéo thì càng gần phân phối chuẩn."""
    (osm, osr), (slope, intercept, r) = stats.probplot(s, dist="norm")
    ax.scatter(osm, osr, s=10, alpha=0.6, color=color, edgecolor="none")
    ax.plot(osm, slope * osm + intercept, color=OI["black"], linewidth=1.5)
    ax.set_title(title); ax.set_xlabel("Phân vị lý thuyết (chuẩn)"); ax.set_ylabel("Phân vị mẫu")
    ax.text(0.04, 0.95, f"R² = {r**2:.3f}", transform=ax.transAxes, va="top", fontsize=11,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec=OI["lightgrey"]))

qq_panel(axes[1, 0], price, OI["sky"], "(c) Q-Q plot: SalePrice gốc")
qq_panel(axes[1, 1], price_log, OI["green"], "(d) Q-Q plot: log(1 + SalePrice)")
axes[1, 0].yaxis.set_major_formatter(price_fmt)

fig.suptitle("Biểu đồ 1. Phân phối giá bán trước và sau biến đổi logarit", fontsize=16, fontweight="bold", y=1.0)
fig.tight_layout()
save_fig(fig, "fig1_saleprice_distribution.png")
plt.show()

# Lưu cột đã biến đổi để dùng cho mô hình sau này
df["SalePrice_log"] = price_log

**Insights:** Giá nhà lệch phải rõ rệt (skewness = 1,74; mean ≈ 180,8K$ cao hơn median 160K$) vì thị trường có một nhóm nhỏ nhà cao cấp kéo dài tới ~755K$. Sau biến đổi log, skewness về gần 0 (−0,01) và R² của Q-Q plot tăng từ 0,876 lên 0,985, nên huấn luyện trên thang log sẽ phù hợp với các mô hình tuyến tính; sai số lúc này có nghĩa là sai lệch theo **phần trăm** giá, đúng cách người mua – bán nghĩ về định giá. *Phản biện:* đuôi trái vẫn còn lệch (vài căn dưới ~30K$) và kurtosis còn 1,5, nên log chưa phải "chuẩn hóa hoàn hảo"; cần kiểm tra phần dư của mô hình thay vì chỉ tin vào biến mục tiêu.

## Biểu đồ 2 – Ma trận tương quan

In [ ]:
# Chọn 10 biến số có |tương quan| với SalePrice lớn nhất (Pearson)
num = df.select_dtypes(include="number").drop(columns=["SalePrice_log"])
corr_target = num.corr()["SalePrice"].drop("SalePrice")
top_feats = corr_target.abs().sort_values(ascending=False).head(10).index.tolist()
print("Top 10 biến tương quan mạnh nhất với SalePrice:")
print(corr_target[top_feats].round(3).to_string())

cols_hm = ["SalePrice"] + top_feats
corr_mat = num[cols_hm].corr()

# Bảng màu phân kỳ xanh dương <-> cam (thân thiện người mù màu, thay cho đỏ/xanh lá)
cmap_div = LinearSegmentedColormap.from_list("cb_div", [OI["blue"], "#FFFFFF", OI["verm"]])
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)    # chỉ hiện tam giác dưới, tránh lặp

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_mat, mask=mask, cmap=cmap_div, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f",
            annot_kws={"size": 10}, square=True, linewidths=0.8, linecolor="white",
            cbar_kws={"shrink": 0.75, "label": "Hệ số tương quan Pearson"}, ax=ax)
ax.grid(False)
ax.tick_params(axis="x", rotation=40); ax.tick_params(axis="y", rotation=0)
plt.setp(ax.get_xticklabels(), ha="right")
for lbl in ax.get_yticklabels() + ax.get_xticklabels():
    if lbl.get_text() == "SalePrice": lbl.set_fontweight("bold")   # nhấn mạnh biến mục tiêu
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_title("Biểu đồ 2. Ma trận tương quan giữa SalePrice và 10 biến số ảnh hưởng nhất", pad=14)
fig.tight_layout()
save_fig(fig, "fig2_correlation_matrix.png")
plt.show()

**Insights:** `Overall Qual` (r = 0,80) và `Gr Liv Area` (r = 0,73) là hai biến giải thích giá mạnh nhất, tiếp theo là nhóm diện tích hầm/gara (r ≈ 0,64–0,66). Bản thân các biến đầu vào lại tương quan rất cao với nhau (`Garage Cars`–`Garage Area` 0,89; `Year Built`–`Garage Yr Blt` 0,86; `1st Flr SF`–`Total Bsmt SF` 0,79), nghĩa là chúng "nói cùng một điều": khi xây mô hình nên giữ một biến đại diện cho mỗi cặp để tránh đa cộng tuyến. *Phản biện:* Pearson chỉ đo quan hệ tuyến tính và không chứng minh nhân quả; ngoài ra `Overall Qual` là điểm đánh giá chủ quan nên có thể đã bị ảnh hưởng bởi chính mức giá.

## Biểu đồ 3 – Tác động của các biến phân loại (chất lượng nhà) lên giá bán

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1.5, 1]})
flier = dict(marker="o", markersize=3, markerfacecolor=OI["grey"], markeredgecolor="none", alpha=0.6)

# --- (a) Overall Qual (1-10): chất lượng vật liệu & hoàn thiện tổng thể ---
ax = axes[0]
order_q = list(range(1, 11))
sns.boxplot(data=df, x="Overall Qual", y="SalePrice", order=order_q, hue="Overall Qual",
            hue_order=order_q, palette=sns.color_palette("viridis", 10), legend=False,
            linewidth=1, flierprops=flier, ax=ax)
med_q = df.groupby("Overall Qual")["SalePrice"].median()
ax.plot(range(len(order_q)), med_q.reindex(order_q).values, color=OI["black"], marker="o",
        markersize=4, linewidth=1.2, alpha=0.8, label="Đường median")
cnt_q = df["Overall Qual"].value_counts()
ymax = df["SalePrice"].max()
for i, q in enumerate(order_q):
    ax.text(i, ymax * 1.04, f"n={cnt_q.get(q, 0)}", ha="center", fontsize=9, color=OI["grey"])
ax.set_ylim(0, ymax * 1.1)
ax.set_title("(a) Chất lượng tổng thể (Overall Qual, thang 1–10)")
ax.set_xlabel("Chất lượng tổng thể (1 = rất kém, 10 = xuất sắc)"); ax.set_ylabel("Giá bán")
ax.yaxis.set_major_formatter(price_fmt); ax.legend(loc="center left", bbox_to_anchor=(0.0, 0.62))

# --- (b) Kitchen Qual: chất lượng bếp ---
ax = axes[1]
# Gộp "Po" (chỉ 1 mẫu) vào "Fa" để hộp không bị suy biến
dfk = df.assign(KQ=df["Kitchen Qual"].replace({"Po": "Fa"}))
order_k = ["Fa", "TA", "Gd", "Ex"]
names_k = {"Fa": "Kém/Tạm", "TA": "Trung bình", "Gd": "Tốt", "Ex": "Xuất sắc"}
pal_k = dict(zip(order_k, sns.color_palette("viridis", len(order_k))))
sns.boxplot(data=dfk, x="KQ", y="SalePrice", order=order_k, hue="KQ",
            hue_order=order_k, palette=pal_k, legend=False, linewidth=1, flierprops=flier, ax=ax)
cnt_k = dfk["KQ"].value_counts()
ax.set_xticks(range(len(order_k)))
ax.set_xticklabels([f"{names_k[o]}\n(n={cnt_k[o]})" for o in order_k])
ax.set_title("(b) Chất lượng bếp (Kitchen Qual)")
ax.set_xlabel("Mức chất lượng bếp"); ax.set_ylabel("")
ax.yaxis.set_major_formatter(price_fmt); ax.set_ylim(0, ymax * 1.1)

fig.suptitle("Biểu đồ 3. Giá bán theo chất lượng tổng thể và chất lượng bếp", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "fig3_categorical_quality.png")
plt.show()

# Số liệu phục vụ nhận xét
print("Median theo Overall Qual:\n", med_q.round(0).astype(int).to_string())
med_k = dfk.groupby("KQ")["SalePrice"].median()
print("\nMedian theo Kitchen Qual:\n", med_k.reindex(order_k).round(0).astype(int).to_string())
print("\nTỷ lệ median Qual 9-10 / Qual 5:", round(med_q.loc[[9, 10]].mean() / med_q.loc[5], 2))

**Insights:** Giá tăng gần như đơn điệu theo chất lượng: median từ 50K$ (mức 1) lên 457,5K$ (mức 10), và nhà mức 9–10 đắt gấp khoảng 3,1 lần nhà mức 5. Riêng bếp "Xuất sắc" có median 331K$, gấp ~2,4 lần bếp "Trung bình" (136,5K$), cho thấy người mua trả giá cao cho chất lượng hoàn thiện, nên đầu tư nâng cấp bếp và hoàn thiện có thể tạo biên lợi nhuận tốt khi bán lại. *Phản biện:* các hộp chồng lấn đáng kể và nhà có bếp xuất sắc thường cũng lớn hơn, chất lượng tổng thể cao hơn, nên khoảng chênh 2,4 lần *không* phải hiệu ứng riêng của bếp; các nhóm cực trị (mức 1–2, mức 10 chỉ có n = 28) cũng có ít mẫu.

## Biểu đồ 4 – Ảnh hưởng của khu vực địa lý (`Neighborhood`)

In [ ]:
# Tính thống kê giá theo khu phố: median, Q1, Q3, số giao dịch, giá/sq ft
df["Price_per_sqft"] = df["SalePrice"] / df["Gr Liv Area"]
nb = df.groupby("Neighborhood").agg(median=("SalePrice", "median"),
                                    q1=("SalePrice", lambda s: s.quantile(0.25)),
                                    q3=("SalePrice", lambda s: s.quantile(0.75)),
                                    n=("SalePrice", "size"),
                                    ppsf=("Price_per_sqft", "median")).sort_values("median")
nb_names = nb.index.tolist()

# Kỹ thuật highlight: Top 3 đắt nhất (cam đỏ), Top 3 rẻ nhất (xanh dương), còn lại xám nhạt
top3, bot3 = nb_names[-3:], nb_names[:3]
col_of = lambda n: OI["verm"] if n in top3 else (OI["blue"] if n in bot3 else OI["lightgrey"])
colors = [col_of(n) for n in nb_names]
ypos = np.arange(len(nb))
city_median = df["SalePrice"].median()
city_ppsf = df["Price_per_sqft"].median()

fig, axes = plt.subplots(1, 2, figsize=(15, 9), sharey=True, gridspec_kw={"width_ratios": [1.5, 1]})

# --- (a) Median giá + khoảng IQR (Q1–Q3) ---
ax = axes[0]
ax.hlines(ypos, nb["q1"], nb["q3"], color=colors, linewidth=5, alpha=0.9, zorder=2)   # khoảng IQR
ax.scatter(nb["median"], ypos, color=colors, s=70, edgecolor="black", linewidth=0.8, zorder=3)  # median
ax.axvline(city_median, color=OI["black"], linestyle="--", linewidth=1.3, zorder=1)
ax.text(city_median + 3000, len(nb) - 0.3, f"Median toàn thành phố: ${city_median/1000:,.0f}K",
        fontsize=10.5, va="bottom")
ax.set_yticks(ypos); ax.set_yticklabels(nb_names)
for lbl, n in zip(ax.get_yticklabels(), nb_names):
    if n in top3 + bot3: lbl.set_fontweight("bold")
ax.set_xlabel("Giá bán: median (chấm) và khoảng IQR Q1–Q3 (thanh)")
ax.set_title("(a) Giá bán theo khu phố")
ax.xaxis.set_major_formatter(price_fmt); ax.grid(axis="y", visible=False)
ax.set_ylim(-0.7, len(nb) + 0.3)
for y, n in zip(ypos, nb["n"]):
    ax.text(ax.get_xlim()[1], y, f" n={n}", va="center", fontsize=8.5, color=OI["grey"])

# --- (b) Median giá / sq ft: kiểm tra ưu thế đến từ "đắt hơn" hay chỉ "to hơn" ---
ax = axes[1]
ax.hlines(ypos, city_ppsf, nb["ppsf"], color=colors, linewidth=2, zorder=2)   # lollipop so với median thành phố
ax.scatter(nb["ppsf"], ypos, color=colors, s=70, edgecolor="black", linewidth=0.8, zorder=3)
ax.axvline(city_ppsf, color=OI["black"], linestyle="--", linewidth=1.3, zorder=1)
ax.set_xlabel("Median giá trên mỗi sq ft ($)")
ax.set_title("(b) Giá trên mỗi sq ft")
ax.grid(axis="y", visible=False)

handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=OI["verm"], markeredgecolor="black", markersize=10, label="3 khu đắt nhất"),
           Line2D([0], [0], marker="o", color="w", markerfacecolor=OI["blue"], markeredgecolor="black", markersize=10, label="3 khu rẻ nhất"),
           Line2D([0], [0], marker="o", color="w", markerfacecolor=OI["lightgrey"], markeredgecolor="black", markersize=10, label="Các khu còn lại")]
axes[0].legend(handles=handles, loc="lower right")

fig.suptitle("Biểu đồ 4. Giá nhà theo khu vực địa lý (Neighborhood)", fontsize=16, fontweight="bold", y=0.96)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_fig(fig, "fig4_neighborhood.png")
plt.show()

# Số liệu phục vụ nhận xét
print(nb[["median", "ppsf", "n"]].round(0).tail(3).to_string())
print(nb[["median", "ppsf", "n"]].round(0).head(3).to_string())
print(f"\nChênh lệch median khu cao nhất / thấp nhất: {nb['median'].iloc[-1]/nb['median'].iloc[0]:.2f}x")
print(f"Chênh lệch $/sq ft cao nhất / thấp nhất: {nb['ppsf'].max()/nb['ppsf'].min():.2f}x")
print(f"Median Gr Liv Area top3 vs bot3: {df[df.Neighborhood.isin(top3)]['Gr Liv Area'].median():.0f} vs {df[df.Neighborhood.isin(bot3)]['Gr Liv Area'].median():.0f}")

**Insights:** Vị trí tạo chênh lệch rất lớn: median của NridgHt, StoneBr, NoRidge (khoảng 302–319K$) gấp ~3,6 lần MeadowV (88K$). Về giá trên mỗi sq ft, NridgHt (160$) và StoneBr (158$) vẫn cao gấp ~1,8 lần MeadowV (86$), tức đây thực sự là khu "đắt hơn"; còn NoRidge chỉ ở mức ~129$/sq ft, nghĩa là giá cao chủ yếu nhờ nhà rất lớn (median diện tích nhóm top 3 là 2.042 sq ft so với 1.092 sq ft ở nhóm rẻ nhất). Ứng dụng: khi định giá hoặc chọn phân khúc đầu tư, phải tách "phí vị trí" khỏi "phí diện tích"; `Neighborhood` nên là một đặc trưng bắt buộc của mô hình. *Phản biện:* các khu có ít giao dịch (GrnHill n = 2, Landmrk n = 1, Greens n = 8) cho số liệu không đáng tin, không nên rút kết luận từ chúng.

## Biểu đồ 5 – Phân tích đa biến: diện tích × chất lượng × gara

In [ ]:
# Gom Overall Qual (1-10) thành 4 nhóm để đọc được màu
bins, labels = [0, 4, 6, 8, 10], ["Thấp (1–4)", "Trung bình (5–6)", "Khá–Tốt (7–8)", "Xuất sắc (9–10)"]
df["Quality_tier"] = pd.cut(df["Overall Qual"], bins=bins, labels=labels)
tier_colors = dict(zip(labels, [OI["sky"], OI["green"], OI["orange"], OI["verm"]]))   # bảng màu Okabe-Ito

# Kích thước điểm tỷ lệ với diện tích gara (ft²)
size_of = lambda g: 12 + g / 1400 * 150

fig, ax = plt.subplots(figsize=(12, 8))
slopes = {}
for tier in labels:
    d = df[df["Quality_tier"] == tier]
    ax.scatter(d["Gr Liv Area"], d["SalePrice"], s=size_of(d["Garage Area"]), alpha=0.5,
               color=tier_colors[tier], edgecolor="white", linewidth=0.4, zorder=2)
    # Đường hồi quy tuyến tính theo từng nhóm: độ dốc = giá trị mỗi sq ft thêm vào
    slope, intercept = np.polyfit(d["Gr Liv Area"], d["SalePrice"], 1)
    slopes[tier] = slope
    xs = np.linspace(d["Gr Liv Area"].min(), d["Gr Liv Area"].max(), 50)
    ax.plot(xs, slope * xs + intercept, color=tier_colors[tier], linewidth=3, zorder=3,
            path_effects=None)

ax.set_xlabel("Diện tích sinh hoạt trên mặt đất – Gr Liv Area (sq ft)")
ax.set_ylabel("Giá bán")
ax.yaxis.set_major_formatter(price_fmt)
ax.set_title("Biểu đồ 5. Giá bán theo diện tích, chất lượng tổng thể và diện tích gara")

# Legend 1: màu = nhóm chất lượng (kèm độ dốc hồi quy)
h1 = [Line2D([0], [0], color=tier_colors[t], linewidth=4, label=f"{t}: +${slopes[t]:,.0f}/sq ft") for t in labels]
leg1 = ax.legend(handles=h1, title="Nhóm chất lượng (độ dốc hồi quy)", loc="upper left")
ax.add_artist(leg1)
# Legend 2: kích thước = diện tích gara
h2 = [Line2D([0], [0], marker="o", color="w", markerfacecolor=OI["grey"], markeredgecolor="white",
             markersize=np.sqrt(size_of(g)), label=f"{g} sq ft") for g in [0, 400, 800, 1200]]
ax.legend(handles=h2, title="Diện tích gara", loc="lower right", labelspacing=1.2, borderpad=0.8)

fig.tight_layout()
save_fig(fig, "fig5_multivariate.png")
plt.show()

# Số liệu phục vụ nhận xét
print({k: round(v) for k, v in slopes.items()})
print(df.groupby("Quality_tier")[["Gr Liv Area", "SalePrice", "Garage Area"]].median().round(0).to_string())
print("Corr(Gr Liv Area, Garage Area) =", round(df["Gr Liv Area"].corr(df["Garage Area"]), 3))
# Giá trung bình của nhà có gara lớn (>=600) so với nhỏ (<400) trong cùng nhóm Khá-Tốt và cùng cỡ diện tích
sub = df[(df["Quality_tier"] == "Khá–Tốt (7–8)") & df["Gr Liv Area"].between(1500, 2000)]
print(sub.groupby(pd.cut(sub["Garage Area"], [-1, 400, 600, 2000]))["SalePrice"].agg(["median", "size"]).round(0))

**Insights:** Mỗi sq ft diện tích thêm vào có giá trị khác nhau tùy chất lượng: độ dốc hồi quy tăng từ +45$/sq ft (nhà chất lượng thấp) lên +125$/sq ft (nhà xuất sắc), gấp ~2,8 lần. Nghĩa là diện tích và chất lượng **tương tác** chứ không cộng gộp đơn giản; mô hình tuyến tính nên có thêm đặc trưng như `Gr Liv Area × Overall Qual`, còn chủ nhà chỉ nên mở rộng diện tích khi mức hoàn thiện đủ cao thì mới được định giá tương xứng. Gara lớn đi kèm giá cao hơn (trong nhóm chất lượng 7–8, 1.500–2.000 sq ft: median ~250K$ khi gara trên 600 sq ft so với ~171K$ khi dưới 400 sq ft). *Phản biện:* gara tương quan với diện tích nhà (r = 0,48) và chất lượng, nên chênh lệch này không thể quy hết cho gara; độ dốc của nhóm 9–10 cũng dựa trên ít điểm ở vùng trên 3.000 sq ft.

## Tổng kết & hướng tiếp theo cho mô hình ML

- **Biến mục tiêu:** dùng `log(1 + SalePrice)` để huấn luyện (cột `SalePrice_log` đã được tạo), sau đó đổi ngược bằng `expm1` khi dự đoán.
- **Biến quan trọng nhất:** `Overall Qual`, `Gr Liv Area`, các biến gara/hầm; `Neighborhood` mang thông tin vị trí độc lập → nên encode (target/one-hot).
- **Đa cộng tuyến:** `Garage Cars`–`Garage Area`, `1st Flr SF`–`Total Bsmt SF`, `TotRms AbvGrd`–`Gr Liv Area` tương quan cao → cân nhắc loại bớt hoặc dùng mô hình có regularization/cây quyết định.
- **Tương tác:** hiệu ứng của diện tích phụ thuộc vào chất lượng (Biểu đồ 5) → thêm đặc trưng tương tác như `Gr Liv Area × Overall Qual`.
- **Chống rò rỉ dữ liệu:** các bước impute thống kê nên đặt trong `Pipeline` và fit chỉ trên tập train.